# Заняття 17 — Потокова обробка даних. Частина 2: stream processing

## Основні цілі заняття:
* Зрозуміти різницю між **batch** і **stream processing** на рівні моделі обчислень.
* Освоїти **Spark Structured Streaming**: `readStream`, `writeStream`, micro-batch модель.
* Прочитати **живий Kafka-топік** із заняття 16 і розпакувати Avro-повідомлення.
* Навчитися працювати з **event time**, **watermark** і **late data** — і побачити, як пізня подія реально відкидається.
* Побачити **output modes** (append / update / complete) у дії.
* Зрозуміти роль **checkpoint** для fault tolerance: зупинити запит, дописати події, перезапустити — і побачити, що обробились лише нові.
* Синтезувати **CDC → SCD2** pipeline і розрізнити три рівні версіонування (CDC / SCD2 / time-travel).
* Завершити серію архітектурним порівнянням **Lambda vs Kappa**.

* **Датасет:** NYC TLC Yellow Taxi Trip Records — той самий датасет, що й у попередніх заняттях. Тут поїздки приходять як потік подій у Kafka.
* **Spark:** local mode, PySpark 4.1.1 (`uv run jupyter lab`)
* **Source:** **живий Kafka-топік `taxi-trips`** зі стеку заняття 16 (`lesson-16-stream-processing-1/code/`). Продюсер із L16 публікує туди поїздки в Avro через Schema Registry — ми читаємо цей самий топік через Spark. Це та сама інфраструктура, що й на минулому занятті: L16 показав producer → broker → consumer, L17 ставить на місце consumer-а Spark.
* **Sink для demo:** **memory sink** (`format("memory")`) для агрегацій — результат лягає в in-memory таблицю, яку опитуємо через `spark.sql(...)`; **parquet sink** із checkpoint для секції про fault tolerance.

Структура ноутбука:
0. Передумови — піднятий Kafka-стек із заняття 16
1. Ініціалізація SparkSession (з Kafka + Avro коннекторами)
2. Batch vs Stream — `isStreaming`
3. Kafka source — сирі повідомлення, Confluent wire format, `from_avro`
4. Windowed aggregation + watermark — tumbling windows у memory sink
5. Output modes — append / update / complete
6. Late data + watermark — що відкидається і чому (на живому потоці)
7. Checkpoint і state management — зупинка та перезапуск запиту
8. CDC → SCD2 synthesis — три рівні версіонування
9. Lambda vs Kappa — архітектурна кульмінація серії
10. Cleanup — зупинка streaming-запитів

## 0. Передумови — Kafka-стек із заняття 16

Цей ноутбук читає **живий топік**, тому перед стартом треба підняти стек L16 і наповнити топік подіями.
Виконайте у терміналі (не в ноутбуці) з каталогу `lesson-16-stream-processing-1/code`:

```bash
cd lesson-16-stream-processing-1/code

# 1. Підняти Kafka (KRaft) + Schema Registry + Kafka UI
docker compose up -d
docker compose ps            # чекаємо, поки kafka і schema-registry стануть healthy (~1 хв)

# 2. Створити топік (auto-create вимкнено навмисно)
docker exec l16-kafka kafka-topics --bootstrap-server localhost:9092 \
  --create --topic taxi-trips --partitions 3 --replication-factor 1

# 3. Наповнити топік поїздками (500 подій, ~5 с)
uv run producer.py
```

**Чистий старт між прогонами.** Демо в секціях 6 і 7 дописує в топік службові події
(`pu_location_id` 999 і 777). Щоб повторний прогін ноутбука був відтворюваним, перестворіть топік:

```bash
docker exec l16-kafka kafka-topics --bootstrap-server localhost:9092 --delete --topic taxi-trips
docker exec l16-kafka kafka-topics --bootstrap-server localhost:9092 \
  --create --topic taxi-trips --partitions 3 --replication-factor 1
uv run producer.py
```

Kafka UI на http://localhost:8082 — там видно топік, партиції, повідомлення і consumer groups.
Наприкінці заняття: `docker compose down -v`.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType, TimestampType, LongType,
)
from pyspark.sql.avro.functions import from_avro
from pyspark.sql.window import Window
from icecream import ic
from datetime import datetime, timedelta
from pathlib import Path
import json
import os
import shutil
import socket
import time

import pyspark

# Kafka stack from lesson 16 — same broker, same topic, same Avro schema.
L16_CODE = Path("../../lesson-16-stream-processing-1/code")
BOOTSTRAP_SERVERS = "127.0.0.1:9092"
SCHEMA_REGISTRY_URL = "http://localhost:8081"
TOPIC = "taxi-trips"
TRIP_SCHEMA = (L16_CODE / "schemas" / "trip.avsc").read_text()

print(TRIP_SCHEMA)

{
  "type": "record",
  "name": "TaxiTrip",
  "namespace": "com.taxi.events",
  "fields": [
    {"name": "pickup_ts",       "type": {"type": "long", "logicalType": "timestamp-millis"}},
    {"name": "dropoff_ts",      "type": {"type": "long", "logicalType": "timestamp-millis"}},
    {"name": "pu_location_id",  "type": ["null", "int"],    "default": null},
    {"name": "do_location_id",  "type": ["null", "int"],    "default": null},
    {"name": "fare_amount",     "type": "double"},
    {"name": "tip_amount",      "type": ["null", "double"], "default": null},
    {"name": "passenger_count", "type": ["null", "int"],    "default": null}
  ]
}



Preflight-перевірка: брокер і Schema Registry відповідають, топік існує і в ньому є повідомлення.
Якщо тут помилка — поверніться до кроків вище, ноутбук далі не працюватиме.

In [2]:
def check_port(host: str, port: int) -> bool:
    with socket.socket() as s:
        s.settimeout(2)
        return s.connect_ex((host, port)) == 0


from confluent_kafka.admin import AdminClient

broker_up = check_port("127.0.0.1", 9092)
registry_up = check_port("127.0.0.1", 8081)

admin = AdminClient({"bootstrap.servers": BOOTSTRAP_SERVERS})
topics = admin.list_topics(timeout=10).topics
partitions = len(topics[TOPIC].partitions) if TOPIC in topics else 0

ic(broker_up, registry_up, TOPIC in topics, partitions)
assert broker_up and registry_up and TOPIC in topics, "Підніміть стек L16 і створіть топік (секція 0)"

ic| broker_up: True
    registry_up: True
    TOPIC in topics: True
    partitions: 3


## 1. Ініціалізація SparkSession

Щоб Spark умів читати Kafka і розпаковувати Avro, потрібні два Maven-пакети — вони не входять у базовий PySpark:

| Пакет | Навіщо |
|---|---|
| `spark-sql-kafka-0-10` | джерело/приймач `format("kafka")` |
| `spark-avro` | функції `from_avro` / `to_avro` |

Суфікс `_2.13` — це версія **Scala**, а не Python: Spark 4.x зібраний під Scala 2.13, і взяти пакет
із суфіксом `_2.12` (як для Spark 3.x) означає `ClassNotFoundException`. Версію самого пакета
беремо рівною версії Spark, тому формуємо рядок із `pyspark.__version__` — так він не роз'їдеться
з `pyproject.toml`.

При першому запуску Spark **завантажує jar-и з Maven Central** (~30 с, потрібен інтернет); далі вони
кешуються в `~/.ivy2`. `shuffle.partitions = 4` робить micro-batch швидшим на ноутбуці (дефолтні 200
партицій — забагато для маленького потоку).

In [3]:
SCALA_VERSION = "2.13"          # Spark 4.x is built for Scala 2.13
SPARK_VERSION = pyspark.__version__

KAFKA_PACKAGES = ",".join([
    f"org.apache.spark:spark-sql-kafka-0-10_{SCALA_VERSION}:{SPARK_VERSION}",
    f"org.apache.spark:spark-avro_{SCALA_VERSION}:{SPARK_VERSION}",
])

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("lesson-17-stream-processing")
    .config("spark.jars.packages", KAFKA_PACKAGES)
    .config("spark.sql.shuffle.partitions", 4)
    .config("spark.sql.repl.eagerEval.enabled", True)
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

ic(spark.version, KAFKA_PACKAGES)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/16 17:40:35 WARN Utils: Your hostname, MacBook-Air-Illia.local, resolves to a loopback address: 127.0.0.1; using 192.168.50.236 instead (on interface en0)
26/09/16 17:40:35 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/Users/illia/projects/UA_DATA-ENGINEERING_KHOROSHYKH-2-hw/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/illia/.ivy2.5.2/cache
The jars for the packages stored in: /Users/illia/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
org.apache.spark#spark-avro_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-47a1c02b-c2c1-46f5-a023-e786d643fcb7;1.0
	confs: [default]


	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.1.1 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.1.1 in central
	found org.apache.kafka#kafka-clients;3.9.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.8 in central
	found org.slf4j#slf4j-api;2.0.17 in central
	found org.apache.hadoop#hadoop-client-runtime;3.4.2 in central
	found org.apache.hadoop#hadoop-client-api;3.4.2 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.scala-lang.modules#scala-parallel-collections_2.13;1.2.0 in central
	found org.apache.commons#commons-pool2;2.12.1 in central
	found org.apache.spark#spark-avro_2.13;4.1.1 in central
	found org.tukaani#xz;1.10 in central
:: resolution report :: resolve 219ms :: artifacts dl 5ms
	:: modules in use:
	com.google.code.findbugs#jsr305;3.0.0 from central in [default]
	org.apache.commons#commons-pool2;2.12.1 from central in [default]
	org.apache.hadoop#hadoop-client-ap

26/09/16 17:40:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


ic| spark.version: '4.1.1'
    KAFKA_PACKAGES: 'org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.1,org.apache.spark:spark-avro_2.13:4.1.1'


('4.1.1',
 'org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.1,org.apache.spark:spark-avro_2.13:4.1.1')

## 2. Batch vs Stream

**Batch processing** обробляє скінченний набір даних (bounded dataset): прочитали файл за місяць, обробили, записали. High latency, запуск за розкладом.

**Stream processing** обробляє нескінченний потік подій (unbounded dataset): результат доступний через секунди після події.

Spark Structured Streaming використовує **micro-batch** модель: кожні N секунд Spark запускає маленьку batch-обробку нових записів. Той самий DataFrame API, що й у batch — різниця лише в source.

Перевірити, потоковий DataFrame чи ні, можна через `.isStreaming`. Статичне читання — `False`.

In [4]:
# Static (batch) read of a tiny in-memory dataset — isStreaming == False
static_df = spark.createDataFrame(
    [(1, "EWR", 12.5), (2, "JFK", 30.0), (3, "LGA", 18.0)],
    schema=["trip_id", "zone", "fare_amount"],
)

ic(static_df.isStreaming)
static_df

ic| static_df.isStreaming: False


trip_id,zone,fare_amount
1,EWR,12.5
2,JFK,30.0
3,LGA,18.0


## 3. Kafka source

### Спершу подивимось на сирі повідомлення — у batch-режимі

Той самий коннектор `format("kafka")` працює і як **batch** source (`spark.read`), і як **streaming**
(`spark.readStream`). Batch-читання зручне для розвідки: воно прочитає діапазон offset-ів і завершиться,
без довгоживучого запиту. Це найпростіший спосіб побачити, що саме лежить у топіку.

Spark завжди повертає з Kafka **фіксовану схему**, незалежно від того, що всередині повідомлення:

| Колонка | Що це |
|---|---|
| `key`, `value` | `binary` — Spark не знає і не хоче знати, що там: JSON, Avro, protobuf |
| `topic`, `partition`, `offset` | координати повідомлення в лозі |
| `timestamp` | час, коли брокер прийняв повідомлення — це **processing time**, а не event time |

Ключовий момент: `timestamp` від Kafka — це **не** час події. Час події (`pickup_ts`) лежить усередині
payload-у, і дістатись до нього можна лише після десеріалізації.

In [5]:
raw_batch = (
    spark.read
    .format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP_SERVERS)
    .option("subscribe", TOPIC)
    .option("startingOffsets", "earliest")
    .option("endingOffsets", "latest")
    .load()
)

raw_batch.printSchema()
ic(raw_batch.count())

(
    raw_batch
    .select(
        F.col("key").cast("string").alias("key"),
        "partition", "offset",
        F.col("timestamp").alias("broker_ts"),   # processing time, NOT event time
        F.length("value").alias("value_bytes"),
    )
    .orderBy("partition", "offset")
    .show(5, truncate=False)
)

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



ic| raw_batch.count(): 500


+---+---------+------+-----------------------+-----------+
|key|partition|offset|broker_ts              |value_bytes|
+---+---------+------+-----------------------+-----------+
|132|0        |0     |2026-09-16 16:59:42.921|41         |
|132|0        |1     |2026-09-16 16:59:42.921|42         |
|132|0        |2     |2026-09-16 16:59:42.921|42         |
|132|0        |3     |2026-09-16 16:59:42.921|42         |
|50 |0        |4     |2026-09-16 16:59:42.921|41         |
+---+---------+------+-----------------------+-----------+
only showing top 5 rows


### Confluent wire format — чому `from_avro` не працює «в лоб»

Продюсер із L16 серіалізує через Schema Registry, а той додає до Avro-payload-у **5-байтовий заголовок**:

```
[ 0x00 ][ schema_id: 4 bytes big-endian ][ Avro binary payload ... ]
  magic          ідентифікатор схеми       власне дані
   byte          у Schema Registry
```

Завдяки цим 5 байтам консюмер знає, **якою саме версією схеми** записане повідомлення, і може витягнути
її з реєстру — це і є механізм schema evolution із заняття 16.

Але `from_avro` в OSS Spark про Schema Registry нічого не знає: він очікує «чистий» Avro і чекає схему
параметром. Тому заголовок треба відрізати вручну — `substring(value, 6, ...)` (індексація в Spark
**з одиниці**, тож 6-й байт = перший байт payload-у). Схему беремо з того самого `.avsc`-файлу, що й
продюсер L16.

> У продакшні цей хак зазвичай ховають у бібліотеку (ABRiS) або користуються Databricks-івським
> `from_avro(..., schemaRegistryAddress)`. Механіка під капотом — рівно ця сама.

In [6]:
# Proof: every message starts with the magic byte 0x00 + a 4-byte schema id
(
    raw_batch
    .select(
        F.hex(F.expr("substring(value, 1, 5)")).alias("wire_header"),
        F.hex(F.expr("substring(value, 6, 8)")).alias("avro_payload_head"),
    )
    .show(3, truncate=False)
)

+-----------+-----------------+
|wire_header|avro_payload_head|
+-----------+-----------------+
|0000000001 |909E90A59863C0B6 |
|0000000001 |D0D692A59863C0E0 |
|0000000001 |A0E692A59863A0F0 |
+-----------+-----------------+
only showing top 3 rows


Тепер — потокове читання. `readStream` замість `read`, і замість `endingOffsets` — параметри потоку:

* `startingOffsets` — звідки починати, якщо checkpoint-у ще немає: `earliest` (весь лог) або `latest` (лише нові).
  **Коли checkpoint існує, цей параметр ігнорується** — позиція береться з нього (секція 7).
* `maxOffsetsPerTrigger` — скільки повідомлень максимум брати в один micro-batch. Тут це передусім
  дидактичний засіб: без нього Spark проковтне всі 500 подій одним batch-ем, і ми не побачимо, як
  watermark рухається поступово. У продакшні це запобіжник від перевантаження після простою.

`kafka_trips()` — одна функція, яку перевикористовуємо в усіх наступних секціях: читає топік,
відрізає 5-байтовий заголовок, розпаковує Avro і піднімає поля запису на верхній рівень.

In [7]:
# Confluent wire format: 1 magic byte + 4 bytes schema id, then the Avro payload.
CONFLUENT_HEADER_BYTES = 5
avro_payload = F.expr(f"substring(value, {CONFLUENT_HEADER_BYTES + 1}, length(value) - {CONFLUENT_HEADER_BYTES})")


def kafka_trips(starting_offsets: str = "earliest", max_offsets_per_trigger: int = 100):
    """Streaming DataFrame of decoded taxi trips from the lesson-16 Kafka topic."""
    raw = (
        spark.readStream
        .format("kafka")
        .option("kafka.bootstrap.servers", BOOTSTRAP_SERVERS)
        .option("subscribe", TOPIC)
        .option("startingOffsets", starting_offsets)
        .option("maxOffsetsPerTrigger", max_offsets_per_trigger)
        .load()
    )
    return (
        raw
        .select(
            from_avro(avro_payload, TRIP_SCHEMA).alias("trip"),
            F.col("timestamp").alias("ingestion_ts"),   # processing time
            "partition",
            "offset",
        )
        .select("trip.*", "ingestion_ts", "partition", "offset")
    )


trips_stream = kafka_trips()

ic(trips_stream.isStreaming)
trips_stream.printSchema()

root
 |-- pickup_ts: timestamp (nullable = true)
 |-- dropoff_ts: timestamp (nullable = true)
 |-- pu_location_id: integer (nullable = true)
 |-- do_location_id: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- ingestion_ts: timestamp (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)



ic| trips_stream.isStreaming: True


Швидка перевірка, що десеріалізація справді працює: ті самі поля, але прочитані в batch-режимі.
`pickup_ts` тут — уже нормальний `timestamp`, а не байти (Avro-логічний тип `timestamp-millis`).

In [8]:
(
    raw_batch
    .select(from_avro(avro_payload, TRIP_SCHEMA).alias("trip"))
    .select("trip.pickup_ts", "trip.pu_location_id", "trip.fare_amount", "trip.passenger_count")
    .orderBy("pickup_ts")
    .show(5, truncate=False)
)

+-------------------+--------------+-----------+---------------+
|pickup_ts          |pu_location_id|fare_amount|passenger_count|
+-------------------+--------------+-----------+---------------+
|2024-01-01 02:00:00|140           |4.4        |1              |
|2024-01-01 02:00:02|161           |8.6        |2              |
|2024-01-01 02:00:03|136           |5.1        |1              |
|2024-01-01 02:00:04|125           |33.8       |1              |
|2024-01-01 02:00:05|132           |70.0       |2              |
+-------------------+--------------+-----------+---------------+
only showing top 5 rows


### Fallback: rate source (якщо стек не піднявся)

Якщо Kafka під час заняття недоступна, будь-яку наступну секцію можна відпрацювати на вбудованому
`rate` source — він генерує `(timestamp, value)` без жодних зовнішніх сервісів. Достатньо підмінити
`trips_stream` наведеним нижче кодом: далі все API ідентичне, бо Structured Streaming не розрізняє
джерела на рівні трансформацій.

Мінус fallback-у: `pickup_ts` там дорівнює processing time, тому демо з late data (секція 6) на ньому
не показати — пізніх подій просто не буває.

In [9]:
# Fallback source — no Kafka needed:
# trips_stream = (
#     spark.readStream.format("rate").option("rowsPerSecond", 5).load()
#     .withColumn("pickup_ts", F.col("timestamp"))
#     .withColumn("pu_location_id", (F.col("value") % 5 + 1).cast("int"))
#     .withColumn("fare_amount", F.round(F.rand(seed=42) * 50 + 5, 2))
#     .select("pickup_ts", "pu_location_id", "fare_amount")
# )

## 4. Windowed aggregation + watermark — серце потокової обробки

Найтиповіша задача: агрегувати події за **часовими вікнами**. Тут — **tumbling window** на 1 хвилину
(вікна не перекриваються) у розрізі `pu_location_id`. Вікна будуємо по `pickup_ts` — **event time**
з payload-у, а не по часу прибуття в Kafka.

**Watermark** — це поріг терпіння до запізнень: `withWatermark("pickup_ts", "2 minutes")` каже Spark
«чекай на пізні події не довше двох хвилин event-time після найпізнішої побаченої події». Без watermark
Spark мусив би тримати state для всіх вікон вічно — бо він не може довести, що пізніх подій більше не буде,
і state ріс би необмежено, поки запит не впаде по пам'яті. Watermark дозволяє закривати старі вікна і
звільняти state.

Дані з L16 — за 1 січня 2024, перші 500 поїздок вкладаються приблизно в 10 хвилин event time. Тому
вікно 1 хвилина + watermark 2 хвилини дають ~10 вікон і помітний рух watermark-а по batch-ах.

Запис — у **memory sink**: результат стає in-memory таблицею `windowed_counts`, яку опитуємо через
`spark.sql(...)`. Output mode — `update`: віддаємо лише ті вікна, що змінились у цьому micro-batch.

In [10]:
WINDOW_DURATION = "1 minute"
WATERMARK_DELAY = "2 minutes"

windowed_agg = (
    trips_stream
    .withWatermark("pickup_ts", WATERMARK_DELAY)
    .groupBy(
        F.window("pickup_ts", WINDOW_DURATION),   # tumbling window (one duration arg)
        F.col("pu_location_id"),
    )
    .agg(
        F.count("*").alias("trip_count"),
        F.round(F.sum("fare_amount"), 2).alias("total_fare"),
    )
)

ic(windowed_agg.isStreaming)

ic| windowed_agg.isStreaming: True


True

Запускаємо streaming-запит. `trigger(processingTime=...)` задає, як часто стартує micro-batch.
Запит **довгоживучий**: він крутиться у фоні, поки ми його явно не зупинимо, — і ми навмисно
залишимо його живим до кінця секції 6, бо саме в нього прилетять пізні події.

In [11]:
windowed_query = (
    windowed_agg.writeStream
    .format("memory")
    .queryName("windowed_counts")
    .outputMode("update")
    .trigger(processingTime="2 seconds")
    .start()
)

time.sleep(15)   # let a few micro-batches run
ic(windowed_query.isActive)

26/09/16 17:40:46 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /private/var/folders/3b/r7klmkpj5sj4vxwp9m43y5x80000gn/T/temporary-10ff4b93-0529-40cf-aac2-acd1d1b2f53a. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/09/16 17:40:46 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


26/09/16 17:40:46 WARN MicroBatchExecution: Disabling AQE since AQE is not supported in stateful workloads.


ic| windowed_query.isActive: True


True

**Пастка memory sink в режимі `update`:** sink лише **дописує** рядки, які віддав кожен micro-batch.
Якщо вікно оновлювалось у трьох batch-ах — у таблиці буде три рядки для цього вікна. Актуальне
значення — останнє, а оскільки лічильники в межах вікна лише ростуть, коректний поточний стан дає
`MAX(...)` з групуванням по вікну і ключу. Це артефакт demo-sink-а, не семантики Spark: справжній
sink (Iceberg, Delta, база) робить upsert по ключу вікна.

Запустіть клітинку кілька разів — побачите, як `trip_count` росте, а нові вікна з'являються.

In [12]:
spark.sql("""
    SELECT window.start AS win_start,
           window.end   AS win_end,
           pu_location_id,
           MAX(trip_count) AS trip_count,     -- dedup the per-micro-batch updates
           MAX(total_fare) AS total_fare
    FROM windowed_counts
    GROUP BY window.start, window.end, pu_location_id
    ORDER BY win_start, trip_count DESC
    LIMIT 10
""")

win_start,win_end,pu_location_id,trip_count,total_fare
2024-01-01 02:00:00,2024-01-01 02:01:00,132,5,285.5
2024-01-01 02:00:00,2024-01-01 02:01:00,161,2,20.7
2024-01-01 02:00:00,2024-01-01 02:01:00,136,1,5.1
2024-01-01 02:00:00,2024-01-01 02:01:00,125,1,33.8
2024-01-01 02:00:00,2024-01-01 02:01:00,229,1,38.0
2024-01-01 02:00:00,2024-01-01 02:01:00,263,1,18.4
2024-01-01 02:00:00,2024-01-01 02:01:00,237,1,5.1
2024-01-01 02:00:00,2024-01-01 02:01:00,113,1,16.3
2024-01-01 02:00:00,2024-01-01 02:01:00,140,1,4.4
2024-01-01 02:00:00,2024-01-01 02:01:00,162,1,14.9


`status`, `lastProgress` і `recentProgress` показують, що відбувається всередині запиту. Найцікавіше:

* `eventTime.watermark` — **поточний watermark**: найпізніший `pickup_ts`, що бачив Spark, мінус 2 хвилини.
* `numInputRows` — скільки подій зайшло в останній micro-batch (обмежено `maxOffsetsPerTrigger`).
* `stateOperators[0].numRowsTotal` — скільки вікон зараз живе в state store.
* `sources[0].endOffset` — до якого offset-у в Kafka дочитали.

У Spark UI це все є на вкладці **Structured Streaming** (http://localhost:4040).

In [13]:
ic(windowed_query.status["message"])
ic(windowed_query.lastProgress["numInputRows"])
ic(windowed_query.lastProgress.get("eventTime"))            # <- watermark lives here
ic(windowed_query.lastProgress["stateOperators"][0]["numRowsTotal"])
ic(windowed_query.lastProgress["sources"][0]["endOffset"])

ic| windowed_query.status["message"]: 'Waiting for next trigger'
ic| windowed_query.lastProgress["numInputRows"]: 0
ic| windowed_query.lastProgress.get("eventTime"): {'watermark': '2024-01-01T00:07:58.000Z'}
ic| windowed_query.lastProgress["stateOperators"][0]["numRowsTotal"]: 105
ic| windowed_query.lastProgress["sources"][0]["endOffset"]: '{'taxi-trips': {'0': 141, '1': 208, '2': 151}}'


"{'taxi-trips': {'0': 141, '1': 208, '2': 151}}"

## 5. Output modes

| Mode | Що записується | Коли використовувати |
|---|---|---|
| **append** | Тільки нові, фінальні рядки | Join, stateless обробка; windowed-агрегації лише ПІСЛЯ закриття вікна watermark-ом |
| **update** | Лише рядки, що змінились у цьому micro-batch | Incremental aggregation з watermark — найчастіший вибір для dashboards |
| **complete** | Весь результат повністю при кожному trigger | Маленькі таблиці зі скінченним набором ключів |

Вище ми використали `update`. Тепер той самий агрегат у режимі **complete** — окремим запитом, поверх
того самого Kafka-топіку. Щоразу memory-таблиця містить ПОВНИЙ знімок усіх вікон, тому тут `MAX(...)`
не потрібен. Зручно для маленьких demo, але не масштабується: у `complete` Spark **ніколи не звільняє
state**, навіть із watermark — результат мусить лишатись повним.

In [14]:
complete_query = (
    kafka_trips()
    .withWatermark("pickup_ts", WATERMARK_DELAY)
    .groupBy(F.window("pickup_ts", WINDOW_DURATION), F.col("pu_location_id"))
    .agg(F.count("*").alias("trip_count"))
    .writeStream
    .format("memory")
    .queryName("windowed_complete")
    .outputMode("complete")
    .trigger(processingTime="2 seconds")
    .start()
)

time.sleep(15)

# complete mode: a full snapshot of every window on every trigger — no dedup needed
spark.sql("SELECT COUNT(*) AS total_window_rows FROM windowed_complete")

26/09/16 17:41:01 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /private/var/folders/3b/r7klmkpj5sj4vxwp9m43y5x80000gn/T/temporary-3d43c5eb-46ea-45be-bd0d-f56f23042b0d. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/09/16 17:41:01 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/09/16 17:41:01 WARN MicroBatchExecution: Disabling AQE since AQE is not supported in stateful workloads.


total_window_rows
268


In [15]:
complete_query.stop()
ic(complete_query.isActive)

26/09/16 17:41:17 WARN DAGScheduler: Failed to cancel job group dff8c111-4622-4b10-bbc2-604a58d87437. Cannot find active jobs for it.
26/09/16 17:41:17 WARN DAGScheduler: Failed to cancel job group dff8c111-4622-4b10-bbc2-604a58d87437. Cannot find active jobs for it.
ic| complete_query.isActive: False


False

**Зауваження про `append` для windowed-агрегацій:** у режимі `append` Spark віддає рядок вікна лише тоді,
коли watermark пройшов `window.end` — тобто вікно вже фінальне і більше не зміниться. Тому в `append`
результати з'являються із затримкою на величину watermark, зате кожен рядок виводиться рівно один раз.
Це обов'язковий режим, коли далі пишемо в append-only sink (наприклад, Iceberg або parquet — як у секції 7).

Звідси практичний наслідок: **watermark — це не лише про пам'ять, це і про latency**. Збільшили watermark
удвічі, щоб не втрачати пізні події, — удвічі збільшили затримку появи результатів в `append`. Це прямий
trade-off completeness проти latency, і вирішується він бізнес-вимогою, а не технічно.

## 6. Late data + watermark — що відкидається і чому

**Event time** — коли подія реально сталась (`pickup_ts` у payload). **Processing time** — коли Spark її
обробив (`ingestion_ts`, він же Kafka `timestamp`). Через GPS-затримки, мережу, офлайн-режим пристрою чи
ретраї продюсера подія з `pickup_ts = 00:02` може дійти до Spark через годину — це **late data**.

Як працює watermark:
```
Найпізніший pickup_ts, що бачив Spark = 00:09:58
watermark = 00:09:58 - 2 min       = 00:07:58
Вікна з window.end <= 00:07:58     → закриті, state звільнено
Подія з pickup_ts < 00:07:58       → ВІДКИНУТА, у результат не потрапить
```

`windowed_query` із секції 4 усе ще працює і вже дочитав топік — його watermark стоїть приблизно на
`00:07:58`. Зараз ми опублікуємо в той самий топік кілька подій із `pickup_ts = 00:00:30` — глибоко за
watermark-ом. Продюсер тут навмисно написаний на тому самому `confluent_kafka` + Schema Registry, що й
`producer.py` з L16: для Kafka це звичайнісінькі повідомлення, «пізніми» їх робить виключно event time
всередині payload-у.

> Той самий ефект можна отримати й з боку L16: у `producer.py` є `LATE_EVENT_FRACTION` (частка пізніх
> подій) і `LATE_EVENT_LAG_MINUTES` (наскільки зсунути їх назад у event time). Поставте
> `LATE_EVENT_FRACTION = 0.1` і перезапустіть продюсер — 10% подій прилетять із запізненням на 5–20 хвилин.

In [16]:
from confluent_kafka import Producer
from confluent_kafka.schema_registry import SchemaRegistryClient
from confluent_kafka.schema_registry.avro import AvroSerializer
from confluent_kafka.serialization import MessageField, SerializationContext

LATE_MARKER_ZONE = 999     # a zone id that cannot appear in real TLC data


def publish_trips(records: list[dict]) -> None:
    """Publish trips to the lesson-16 topic, Avro-encoded via Schema Registry."""
    serializer = AvroSerializer(SchemaRegistryClient({"url": SCHEMA_REGISTRY_URL}), TRIP_SCHEMA)
    producer = Producer({"bootstrap.servers": BOOTSTRAP_SERVERS})
    for record in records:
        producer.produce(
            topic=TOPIC,
            key=str(record["pu_location_id"]),
            value=serializer(record, SerializationContext(TOPIC, MessageField.VALUE)),
        )
    producer.flush()


def trip_record(pickup: datetime, zone: int, fare: float = 99.0) -> dict:
    pickup_ms = int(pickup.timestamp() * 1000)
    return {
        "pickup_ts": pickup_ms,
        "dropoff_ts": pickup_ms + 600_000,
        "pu_location_id": zone,
        "do_location_id": 1,
        "fare_amount": fare,
        "tip_amount": 0.0,
        "passenger_count": 1,
    }


# Event time 00:00:30 — far behind the watermark the running query has reached.
late_events = [trip_record(datetime(2024, 1, 1, 0, 0, 30), LATE_MARKER_ZONE) for _ in range(5)]
publish_trips(late_events)

ic(len(late_events), "published")

ic| len(late_events): 5, 'published'


(5, 'published')

Дамо запиту зробити кілька micro-batch-ів і подивимось на дві речі:

1. `numRowsDroppedByWatermark` у `stateOperators` — Spark **рахує** відкинуті рядки (зверніть увагу:
   рахуються рядки вже після агрегації, тому 5 подій однієї зони дадуть менше 5).
2. Чи з'явилась зона 999 у результаті — не повинна.

Якщо ви **перезапускали** ноутбук і watermark ще не встиг дійти до 00:07:58 — події не відкинуться,
а просто потраплять у своє вікно. Це не баг: пізня подія відкидається тільки тоді, коли її вікно вже закрите.

In [17]:
time.sleep(10)

dropped = sum(
    op["numRowsDroppedByWatermark"]
    for progress in windowed_query.recentProgress
    for op in progress.get("stateOperators", [])
)

ic(windowed_query.lastProgress["eventTime"]["watermark"])
ic(dropped)
ic(spark.sql(f"SELECT COUNT(*) AS c FROM windowed_counts WHERE pu_location_id = {LATE_MARKER_ZONE}").collect()[0]["c"])

ic| windowed_query.lastProgress["eventTime"]["watermark"]: '2024-01-01T00:07:58.000Z'
ic| dropped: 1
ic| spark.sql(f"SELECT COUNT(*) AS c FROM windowed_counts WHERE pu_location_id = {LATE_MARKER_ZONE}").collect()[0]["c"]: 0


0

Зона 999 у результаті відсутня, а лічильник `numRowsDroppedByWatermark` — ненульовий. Це і є ціна watermark-а:
**дані мовчки губляться**. Spark не кидає помилку і не пише попередження в лог — лише збільшує метрику.

Практичний висновок: `numRowsDroppedByWatermark` треба **виносити в моніторинг**. Ненульове значення означає,
що ваше припущення про максимальне запізнення не відповідає реальності — або подовжуйте watermark, або
приймайте втрату свідомо. Якщо пізні дані втрачати не можна взагалі — потрібен окремий маршрут: писати
такі події в side-output і доганяти їх batch-процесом (типовий Lambda-патерн із секції 9).

In [18]:
windowed_query.stop()
ic(windowed_query.isActive)

26/09/16 17:41:27 WARN DAGScheduler: Failed to cancel job group c6626b27-5557-435c-9576-5009ed5d7bba. Cannot find active jobs for it.
26/09/16 17:41:27 WARN DAGScheduler: Failed to cancel job group c6626b27-5557-435c-9576-5009ed5d7bba. Cannot find active jobs for it.
ic| windowed_query.isActive: False


False

## 7. Checkpoint і state management

`checkpointLocation` — обов'язковий для будь-якого streaming-запису, окрім demo-sink-ів (memory / console).
Саме тому в секціях 4–6 ми його не вказували. Checkpoint зберігає три речі:

| Каталог | Що всередині |
|---|---|
| `offsets/` | **WAL позицій у source**: для кожного batch-у — до якого offset-у в кожній партиції Kafka планується дочитати. Записується ДО обробки. |
| `commits/` | Підтвердження, що batch завершився успішно. Batch, у якого є `offsets`, але немає `commits`, після рестарту переграється. |
| `state/` | Знімки state store — стан агрегацій (лічильники по вікнах) і сам watermark. |
| `sources/`, `metadata` | Метадані запиту: його id, версія, опис source-ів. |

Пара `offsets` + `commits` і дає **exactly-once** для файлових sink-ів: Spark знає, який batch не
доїхав, переграє рівно його і атомарно комітить файли через `_spark_metadata`.

Зараз запишемо Bronze-шар у parquet із checkpoint-ом — це вже не demo-sink, а те, як streaming-job
виглядає в проді.

In [19]:
BRONZE_PATH = "data/bronze/trips"
CHECKPOINT_PATH = "data/checkpoints/bronze_trips"

# Fresh start for the demo — in production you would NEVER delete these.
shutil.rmtree("data/bronze", ignore_errors=True)
shutil.rmtree("data/checkpoints", ignore_errors=True)


def start_bronze_query():
    """Kafka -> parquet Bronze. Same code on first run and on restart."""
    return (
        kafka_trips(starting_offsets="earliest", max_offsets_per_trigger=200)
        .writeStream
        .format("parquet")
        .outputMode("append")
        .option("path", BRONZE_PATH)
        .option("checkpointLocation", CHECKPOINT_PATH)
        .trigger(processingTime="2 seconds")
        .start()
    )


bronze_query = start_bronze_query()
time.sleep(20)

rows_first_run = spark.read.parquet(BRONZE_PATH).count()
ic(rows_first_run)
bronze_query.stop()

26/09/16 17:41:27 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


ic| rows_first_run: 505
26/09/16 17:42:57 WARN DAGScheduler: Failed to cancel job group 25c269d5-7070-45f0-b130-fe5a91833f7c. Cannot find active jobs for it.
26/09/16 17:42:57 WARN DAGScheduler: Failed to cancel job group 25c269d5-7070-45f0-b130-fe5a91833f7c. Cannot find active jobs for it.


Заглянемо всередину checkpoint-у. Файл `offsets/<batch_id>` — це звичайний текст: рядок версії, рядок
конфігів запиту і **JSON із позиціями по партиціях**. Саме цей рядок і є «пам'яттю» job-у про те,
де він зупинився.

In [20]:
ic(sorted(p.name for p in Path(CHECKPOINT_PATH).iterdir() if not p.name.startswith(".")))

offset_files = sorted(
    (p for p in Path(CHECKPOINT_PATH, "offsets").iterdir() if p.name.isdigit()),
    key=lambda p: int(p.name),
)
ic([p.name for p in offset_files])
ic(sorted(p.name for p in Path(CHECKPOINT_PATH, "commits").iterdir() if p.name.isdigit()))

# The last line of an offsets file is the Kafka position: {topic: {partition: offset}}
last_offsets = offset_files[-1].read_text().strip().splitlines()[-1]
ic(json.loads(last_offsets))

ic| sorted(p.name for p in Path(CHECKPOINT_PATH).iterdir() if not p.name.startswith(".")): ['commits', 'metadata', 'offsets', 'sources']
ic| [p.name for p in offset_files]: ['0', '1', '2']
ic| sorted(p.name for p in Path(CHECKPOINT_PATH, "commits").iterdir() if p.name.isdigit()): ['0', '1', '2']
ic| json.loads(last_offsets): {'taxi-trips': {'0': 141, '1': 213, '2': 151}}


{'taxi-trips': {'0': 141, '1': 213, '2': 151}}

Тепер найголовніше про checkpoint. Допишемо в топік 20 нових подій і **перезапустимо той самий запит
тим самим кодом** — з `startingOffsets = "earliest"`.

Наївне очікування: запит перечитає весь топік із нуля і продублює Bronze. Реальність: `startingOffsets`
діє **лише коли checkpoint порожній**. Оскільки checkpoint є, Spark бере позицію з нього і обробляє
рівно нові 20 подій. Це і є fault tolerance: впав job, ноутбук, цілий кластер — після старту
продовжуємо з тієї самої точки, без втрат і без дублікатів.

In [21]:
RESTART_MARKER_ZONE = 777
new_events = [
    trip_record(datetime(2024, 1, 1, 0, 20, 0) + timedelta(seconds=i), RESTART_MARKER_ZONE, fare=10.0)
    for i in range(20)
]
publish_trips(new_events)

bronze_query = start_bronze_query()     # same checkpoint, same startingOffsets="earliest"
time.sleep(15)

rows_processed_on_restart = sum(p["numInputRows"] for p in bronze_query.recentProgress)
rows_after_restart = spark.read.parquet(BRONZE_PATH).count()

ic(rows_first_run)                # everything that was in the topic
ic(rows_processed_on_restart)     # only the NEW events — not the whole topic
ic(rows_after_restart)            # rows_first_run + the new events, no duplicates

bronze_query.stop()

26/09/16 17:42:57 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


ic| rows_first_run: 505
ic| rows_processed_on_restart: 20
ic| rows_after_restart: 525
26/09/16 17:43:12 WARN DAGScheduler: Failed to cancel job group a6006ddb-8095-45da-93f3-264f6a66154c. Cannot find active jobs for it.
26/09/16 17:43:12 WARN DAGScheduler: Failed to cancel job group a6006ddb-8095-45da-93f3-264f6a66154c. Cannot find active jobs for it.


**Коли checkpoint ламається.** Checkpoint жорстко прив'язаний до плану запиту. Зміните ключі
групування, схему агрегації, додасте stateful-оператор — і Spark на старті впаде з помилкою
несумісності. Лікування одне: видалити checkpoint, а це означає повний replay source-а
(`startingOffsets = "earliest"`) і, для non-idempotent sink-ів, ручне прибирання вже записаного.

Звідси три робочі правила:
1. **Checkpoint — на надійному сховищі** (S3/HDFS), не в `/tmp` і не поруч із ноутбуком.
2. **Один checkpoint = один запит.** Два запити на один каталог — гарантована корупція стану.
3. **Зміну логіки планують як міграцію**: новий checkpoint + новий вихідний шлях, потім перемикання
   споживачів. Kappa-архітектура (секція 9) саме тому й зручна: replay топіку з нуля — штатна операція,
   а не аварія.

## 8. CDC → SCD2 synthesis — синтез усього курсу

Тут сходяться три заняття: **CDC** (L16 — потік змін із джерела), **SCD2** (L06 — версіонування вимірів у warehouse) і **time-travel** (L10 — Iceberg snapshots). Споживаємо CDC-потік через Structured Streaming і матеріалізуємо його у SCD2-вимір `dim_location`.

### Три рівні версіонування — критична дисемблюація

| Рівень | Питання, на яке відповідає | Механізм |
|---|---|---|
| **CDC** (transport) | Що саме змінилось у джерелі і коли? | Потік row-level змін (insert / update / delete) з Kafka |
| **SCD2** (serving) | Яким був атрибут сутності НА МОМЕНТ факту? (as-of join) | Рядки з `valid_from` / `valid_to` / `is_current` у dim-таблиці |
| **Iceberg time-travel** (storage) | Як виглядала ВСЯ таблиця на коміті X? | Snapshot/версія всього файлу таблиці |

**Iceberg time-travel НЕ замінює SCD2.** Time-travel відповідає на питання про стан таблиці у момент коміту (операційне/аудит). SCD2 відповідає на бізнес-питання «який тариф зони діяв, коли сталась ця поїздка» через as-of join факту з виміром. Якщо переплутати — warehouse не зможе відповідати на as-of бізнес-питання.

In [22]:
# CDC stream (reference): кожне повідомлення — row-level зміна з джерела.
# Формат у стилі Debezium: op (c/u/d), after.<fields>, ts_ms.
# cdc_stream = (
#     spark.readStream.format("kafka")
#     .option("kafka.bootstrap.servers", "localhost:9092")
#     .option("subscribe", "pg.public.location")
#     .load()
#     .select(F.from_json(F.col("value").cast("string"), cdc_schema).alias("c"))
#     .select("c.op", "c.after.location_id", "c.after.zone", "c.after.borough", "c.ts_ms")
# )
#
# Матеріалізація SCD2 робиться у foreachBatch (бо MERGE не працює в чистому stream API):
# def upsert_scd2(batch_df, batch_id):
#     batch_df.createOrReplaceTempView("changes")
#     spark.sql("""
#         MERGE INTO gold.dim_location t
#         USING changes s
#         ON t.location_id = s.location_id AND t.is_current = true
#         WHEN MATCHED AND t.zone <> s.zone THEN
#             UPDATE SET t.is_current = false, t.valid_to = s.event_ts
#     """)
#     # потім окремий INSERT нових версій рядків ...
#
# cdc_stream.writeStream.foreachBatch(upsert_scd2).start()

Щоб показати РЕЗУЛЬТАТ SCD2 без живого Kafka, зматеріалізуємо вимір локально з невеликого набору CDC-подій. Зона `132` (JFK) спершу в Queens, потім «перейменована» — отримуємо дві версії рядка з `valid_from` / `valid_to` / `is_current`.

In [23]:
cdc_events = spark.createDataFrame(
    [
        # (location_id, zone, event_ts, op)
        (132, "JFK Airport",      "2024-01-01 00:00:00", "c"),
        (236, "Upper East Side",  "2024-01-01 00:00:00", "c"),
        (132, "JFK Intl Airport", "2024-03-15 00:00:00", "u"),  # rename -> new SCD2 version
    ],
    schema=["location_id", "zone", "event_ts_str", "op"],
).withColumn("event_ts", F.to_timestamp("event_ts_str")).drop("event_ts_str")

# Build SCD2: valid_from = this version's event_ts; valid_to = next version's event_ts (per key)
w = Window.partitionBy("location_id").orderBy("event_ts")
dim_location = (
    cdc_events
    .withColumn("valid_from", F.col("event_ts"))
    .withColumn("valid_to", F.lead("event_ts").over(w))
    .withColumn("is_current", F.col("valid_to").isNull())
    .select("location_id", "zone", "valid_from", "valid_to", "is_current")
    .orderBy("location_id", "valid_from")
)
dim_location

location_id,zone,valid_from,valid_to,is_current
132,JFK Airport,2024-01-01 00:00:00,2024-03-15 00:00:00,false
132,JFK Intl Airport,2024-03-15 00:00:00,NULL,true
236,Upper East Side,2024-01-01 00:00:00,NULL,true


**As-of join** — навіщо SCD2 існує. Факт-поїздка з `132` 1 лютого має приєднати ту версію зони, що діяла САМЕ ТОДІ («JFK Airport»), а не поточну. Це питання, на яке time-travel не відповідає.

In [24]:
facts = spark.createDataFrame(
    [(1, 132, "2024-02-01 09:00:00"), (2, 132, "2024-04-01 09:00:00")],
    schema=["trip_id", "pu_location_id", "pickup_ts_str"],
).withColumn("pickup_ts", F.to_timestamp("pickup_ts_str")).drop("pickup_ts_str")

asof = (
    facts.join(
        dim_location,
        on=(facts.pu_location_id == dim_location.location_id)
        & (facts.pickup_ts >= dim_location.valid_from)
        & ((facts.pickup_ts < dim_location.valid_to) | dim_location.valid_to.isNull()),
        how="left",
    )
    .select("trip_id", "pickup_ts", "pu_location_id", "zone")
    .orderBy("trip_id")
)
asof

trip_id,pickup_ts,pu_location_id,zone
1,2024-02-01 09:00:00,132,JFK Airport
2,2024-04-01 09:00:00,132,JFK Intl Airport


## 9. Lambda vs Kappa — архітектурна кульмінація

Тепер, коли ви бачили і batch (L11–15), і streaming (L16–17), ці дві архітектури нарешті стають конкретними.

### Lambda Architecture
```
              Джерело подій
                   │
        ┌──────────┴───────────┐
        ▼                      ▼
   Batch Layer            Speed Layer
   (Spark batch)          (Spark Streaming)
   bronze_batch           bronze_stream
   full reprocess         real-time, low-lat
        │                      │
        └──────────┬───────────┘
                   ▼
          Serving Layer (Silver)
          UNION ALL + dedup
```
- `bronze.yellow_trips` — batch Bronze (Path B, щомісяця).
- `bronze.yellow_trips_stream` — streaming Bronze (Kafka → Iceberg, безперервно).
- Silver = `UNION ALL` обох + `dropDuplicates` → serving layer.
- **Проблема:** два codebase для одного результату → maintenance nightmare, batch і stream логіка розходяться.

### Kappa Architecture
```
              Джерело подій
                   │
            Streaming Layer
        (Kafka + Spark/Flink)
            Single pipeline
                   │
            Storage (Iceberg)
       Batch queries = той самий table
```
- Один pipeline, одна codebase. Batch backfill = replay Kafka topic з offset 0 через ТОЙ САМИЙ streaming job (`startingOffsets = earliest`).
- **Переваги:** простота; Iceberg нативно підтримує concurrent batch + streaming writes.
- **Обмеження:** потрібен достатній Kafka retention для повного replay.

### Що ми будуємо в проєкті і чому
- **Педагогічний Lambda** у L17: два окремих фізичних Bronze (speed + batch) і unifying Silver — це teaching device, щоб Lambda стала конкретною.
- **Production-рекомендація:** у реальному проді **Kappa** — правильний вибір. Iceberg підтримує concurrent batch + streaming writes, окрема speed layer не потрібна, backfill = streaming job з `earliest`.

In [25]:
# Serving layer (reference): Lambda reconciliation — UNION batch + stream Bronze, dedup.
# batch_bronze  = spark.read.format("iceberg").load("bronze.yellow_trips")
# stream_bronze = spark.read.format("iceberg").load("bronze.yellow_trips_stream")
#
# w = Window.partitionBy("pu_location_id", "pickup_ts", "fare_amount").orderBy(F.desc("ingestion_ts"))
# silver = (
#     batch_bronze.unionByName(stream_bronze)
#     .withColumn("rn", F.row_number().over(w))
#     .filter(F.col("rn") == 1)
#     .drop("rn")
# )
# silver.writeTo("silver.yellow_trips").overwritePartitions()

## 10. Cleanup

Streaming-запити довгоживучі. Перед завершенням ноутбука зупиняємо всі активні запити, інакше Spark-сесія «висітиме». `spark.streams.active` показує всі активні запити.

In [26]:
for q in spark.streams.active:
    ic(q.name)
    q.stop()

ic(len(spark.streams.active))

ic| len(spark.streams.active): 0


0

In [27]:
spark.stop()